In [ ]:
# .xlsx data file not included in repo, contact to request

import pandas as pd
df = pd.read_excel(r"data/manscirep_meta_data.xlsx")
print(df.columns.tolist()) 
df

####################### API Data Collection (for citations_count, accepted_by_name, accepted_by_field)

In [ ]:
## API Data Collection "citations_count" - Single - TEST

import requests

doi = "10.1287/mnsc.2020.3901"
url = f"https://api.crossref.org/works/{doi}"

response = requests.get(url)
data = response.json()

citations = data["message"]["is-referenced-by-count"]
print("Citations:", citations)

In [ ]:
##### API Data Collection "citations_count", "accepted_by_name", "accepted_by_field" - Iterated - FINAL

import pandas as pd
import requests
import re

df = pd.read_excel(r"data/manscirep_meta_data.xlsx")

df["citations_count"] = None
df["accepted_by_name"] = None
df["accepted_by_field"] = None

for index, row in df.iterrows():
    doi = row["PaperDOI"]
    url = f"https://api.crossref.org/works/{doi}"
    
    print(f"Row {index}: DOI = '{doi}'")  
    
    response = requests.get(url)
    print(f"  Status: {response.status_code}")
    
    if response.status_code == 200:
    
        data = response.json()
        message = data["message"]

        #citations_count

        citations_count = data["message"]["is-referenced-by-count"]
        df.at[index, "citations_count"] = citations_count
        print(f"  Citations: {citations_count}")

        # accepted_by_name / accepted_by_field

        abstract = message.get("abstract", "")
        match = re.search(r"accepted by ([A-Za-z\s\.\-]+),\s*([a-zA-Z\s]+)\.", abstract)

        if match:
            name = match.group(1).strip()
            field = match.group(2).strip()
            df.at[index, "accepted_by_name"] = name
            df.at[index, "accepted_by_field"] = field
            print(f"  Accepted by: {name} ({field})")
        else:
            df.at[index, "accepted_by_name"] = "NA"
            df.at[index, "accepted_by_field"] = "NA"
            print(f"  Accepted by: not found in abstract")

    elif response.status_code == 404:
        print(f"  DOI not found in Crossref")
        df.at[index, "citations_count"] = "NA"
        df.at[index, "accepted_by_name"] = "NA"
        df.at[index, "accepted_by_field"] = "NA"

print(df[["citations_count", "accepted_by_name", "accepted_by_field"]].head(8))
df.to_excel(r"data/manscirep_meta_data_v2.xlsx", index=False)
df

####################### Web Scraping (for downloads_count_str)

In [ ]:
## Web Scraping "citations_count_str", "downloads_count_str" - Single - TEST

from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.microsoft import EdgeChromiumDriverManager

driver = webdriver.Edge(service=Service(EdgeChromiumDriverManager().install()))
driver.get("https://doi.org/10.1287/mnsc.2018.3103")

wait = WebDriverWait(driver, 0.3)

try:
    citations_count_str = driver.find_element(By.ID, "ref_count")
    print("Citations Text:", citations_count_str.text)
except:
    print("Citations count: NA")


try:
    downloads_count_str = driver.find_element(By.CLASS_NAME, "download-count-container")
    print("Downloads count:", downloads_count_str.text)
except:
    print("Downloads count: NA")


driver.quit()

In [ ]:
#### Web Scraping "citations_count_str", "downloads_count_str" - Iterated - FINAL

# only runs in .py file, not in .ipynb
# check repo folder with doi_webscraping_cloakbrowser.py file for script